In [ ]:
# ==========================================
# 實作 08 - NMS 驗證版 (多閾值掃描 + TTA + CBAM)
# ==========================================

import os
import sys
import cv2
import csv
import torch
import numpy as np
import importlib
from tqdm import tqdm
from google.colab import drive

# ==========================================
# 1. 掛載與安裝
# ==========================================
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
%cd /content
if not os.path.exists('/content/yolov7'):
    !git clone https://github.com/ws6125/yolov7.git
%cd yolov7
!pip install -q -r requirements.txt

# 💡 確保權重檔存在並複製
WEIGHTS_SRC = "/content/drive/MyDrive/實作08pt/08_150_best.pt"
if not os.path.exists(WEIGHTS_SRC):
    raise FileNotFoundError(f"❌ 找不到權重檔：{WEIGHTS_SRC}")
!cp "{WEIGHTS_SRC}" ./best.pt

# ==============================================================================
# 2. 注入 CBAM 注意力機制 (維持實作 08 架構)
# ==============================================================================
cbam_code = """
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.f1 = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu = nn.ReLU()
        self.f2 = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.f2(self.relu(self.f1(self.avg_pool(x))))
        max_out = self.f2(self.relu(self.f1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        assert kernel_size in (3, 7), 'kernel size must be 3 or 7'
        padding = 3 if kernel_size == 7 else 1
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x))

class CBAM(nn.Module):
    def __init__(self, c1, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(c1, ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        out = self.channel_attention(x) * x
        out = self.spatial_attention(out) * out
        return out
"""
common_path = 'models/common.py'
with open(common_path, 'r') as f:
    content = f.read()
if 'class CBAM' not in content:
    with open(common_path, 'a') as f:
        f.write("\n\n" + cbam_code)

# ==============================================================================
# 3. 破解限制與重載模組
# ==============================================================================
general_path = 'utils/general.py'
with open(general_path, 'r') as f: content = f.read()
content = content.replace('max_det = 300', 'max_det = 10000')
content = content.replace('max_nms = 30000', 'max_nms = 100000')
with open(general_path, 'w') as f: f.write(content)

if '/content/yolov7' not in sys.path: sys.path.append('/content/yolov7')
import models.common
importlib.reload(models.common)
import utils.general
importlib.reload(utils.general)

from utils.general import non_max_suppression, scale_coords
from models.experimental import attempt_load
from utils.datasets import letterbox

# ==========================================
# 4. 參數設定區 (多閾值掃描)
# ==========================================
WEIGHTS = './best.pt'
IMAGE_DIR = '/content/drive/MyDrive/testset/images'

IMG_SIZE = 3200
CONF_LIST = [0.55, 0.60, 0.65, 0.70, 0.75, 0.78, 0.80]
IOU_THRES = 0.35
TARGET_CLASS = 2

MIN_AREA = 80

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# ==========================================
# 5. 開始推論並批次寫入 CSV
# ==========================================
print("🔄 正在載入 YOLOv7 模型...")
model = attempt_load(WEIGHTS, map_location=device)
model.eval()
stride = int(model.stride.max())

print(f"🚀 開始推論 (已開啟 TTA，使用傳統 NMS 進行多閾值平行掃描)...")

csv_files = {}
csv_writers = {}
stats = {}

for conf in CONF_LIST:
    filename = f'output_08_NMS_conf_{conf:.2f}.csv'
    f = open(filename, mode='w', newline='')
    writer = csv.writer(f)
    writer.writerow(["ID", "bbox"])
    csv_files[conf] = f
    csv_writers[conf] = writer
    stats[conf] = {'kept': 0, 'removed': 0}

# 支援多種圖片格式
valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(valid_exts)])

for img_name in tqdm(image_files):
    img_path = os.path.join(IMAGE_DIR, img_name)
    img0 = cv2.imread(img_path)
    if img0 is None: continue

    img = letterbox(img0, IMG_SIZE, stride=stride, auto=False)[0]
    img = img[:, :, ::-1].transpose(2, 0, 1)
    img = np.ascontiguousarray(img)

    img_tensor = torch.from_numpy(img).to(device).float() / 255.0
    if img_tensor.ndimension() == 3:
        img_tensor = img_tensor.unsqueeze(0)

    # TTA 預測 (只跑一次，節省時間)
    with torch.no_grad():
        pred_raw = model(img_tensor, augment=True)[0]

    # 對每一個信心度閾值，執行一次獨立的 NMS
    for conf in CONF_LIST:
        # 💡 呼叫官方 NMS (注意：傳入的預測張量必須是 clone，否則 NMS 會修改原始資料)
        pred = non_max_suppression(
            pred_raw.clone(),
            conf,
            IOU_THRES,
            classes=[TARGET_CLASS]
        )

        crd = []
        for det in pred:
            if len(det):
                det[:, :4] = scale_coords(img_tensor.shape[2:], det[:, :4], img0.shape).round()

                for *xyxy, c, cls in det:
                    x_min, y_min = int(xyxy[0]), int(xyxy[1])
                    x_max, y_max = int(xyxy[2]), int(xyxy[3])
                    area = (x_max - x_min) * (y_max - y_min)

                    if area < MIN_AREA:
                        stats[conf]['removed'] += 1
                        continue

                    stats[conf]['kept'] += 1
                    crd.append(f"{x_min} {y_min} {x_max} {y_max}")

        bbox_string = " ".join(crd) if len(crd) > 0 else "0"
        csv_writers[conf].writerow([img_name, bbox_string])

for f in csv_files.values():
    f.close()


print(f"\n✅ 全部完成！限制解除、TTA 開啟、傳統 NMS 已完美套用！")
print("📊 NMS 各信心度閾值 (Confidence Threshold) 成果統計：")
for conf in CONF_LIST:
    print(f"  ➤ Conf: {conf:.2f} | 保留車輛: {stats[conf]['kept']} 個 | 過濾雜訊: {stats[conf]['removed']} 個 | 輸出檔: {filename}")

Mounted at /content/drive
/content
Cloning into 'yolov7'...
remote: Enumerating objects: 943, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 943 (delta 12), reused 9 (delta 9), pack-reused 922 (from 2)
Receiving objects: 100% (943/943), 57.73 MiB | 28.22 MiB/s, done.
Resolving deltas: 100% (420/420), done.
/content/yolov7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 99.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.66.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.21.2 which is incompatible.
google-cloud-datastore 2.24.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.21.2 which is incompatible.
google-cloud-appengine-logging 1.9.0 requires

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


🚀 開始推論 (已開啟 TTA，使用傳統 NMS 進行多閾值平行掃描)...


100%|██████████| 53/53 [03:02<00:00,  3.44s/it]



✅ 全部完成！限制解除、TTA 開啟、傳統 NMS 已完美套用！
📊 NMS 各信心度閾值 (Confidence Threshold) 成果統計：
  ➤ Conf: 0.55 | 保留車輛: 17708 個 | 過濾雜訊: 6 個 | 輸出檔: output_08_NMS_conf_0.80.csv
  ➤ Conf: 0.60 | 保留車輛: 16199 個 | 過濾雜訊: 2 個 | 輸出檔: output_08_NMS_conf_0.80.csv
  ➤ Conf: 0.65 | 保留車輛: 14675 個 | 過濾雜訊: 1 個 | 輸出檔: output_08_NMS_conf_0.80.csv
  ➤ Conf: 0.70 | 保留車輛: 12895 個 | 過濾雜訊: 0 個 | 輸出檔: output_08_NMS_conf_0.80.csv
  ➤ Conf: 0.75 | 保留車輛: 10732 個 | 過濾雜訊: 0 個 | 輸出檔: output_08_NMS_conf_0.80.csv
  ➤ Conf: 0.78 | 保留車輛: 9093 個 | 過濾雜訊: 0 個 | 輸出檔: output_08_NMS_conf_0.80.csv
  ➤ Conf: 0.80 | 保留車輛: 7830 個 | 過濾雜訊: 0 個 | 輸出檔: output_08_NMS_conf_0.80.csv
